# PyTorch自动梯度计算
1. 两个参数和两百万个参数自动求导
2. 手动把梯度置为0
3. PyTorch优化器torch.optim更新参数

In [15]:
import numpy as np
import torch
torch.set_printoptions(edgeitems=2,linewidth=75)
import torch.optim as optim
dir(optim)

['ASGD',
 'Adadelta',
 'Adagrad',
 'Adam',
 'AdamW',
 'Adamax',
 'LBFGS',
 'NAdam',
 'Optimizer',
 'RAdam',
 'RMSprop',
 'Rprop',
 'SGD',
 'SparseAdam',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_functional',
 '_multi_tensor',
 'lr_scheduler',
 'swa_utils']

In [4]:
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0] 
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

In [5]:
t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

In [6]:
def model(t_u,w,b):
    return t_u*w+b
def loss_fn(t_p, t_c):
    squared_diff = (t_p-t_c) **2
    return squared_diff.mean()

In [8]:
params = torch.tensor([1.0,0.0], requires_grad=True)

In [11]:
loss = loss_fn(model(t_u, *params),t_c)
loss.backward()
params.grad

tensor([4517.2969,   82.6000])

tensor([ 9.5483e-01, -8.2600e-04], requires_grad=True)

In [19]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1 ):

        if params.grad is not None:
            params.grad.zero_()
        t_p = model (t_u, *params)
        loss = loss_fn(t_p, t_c)
        loss.backward()

        with torch.no_grad():
            params -=learning_rate * params.grad
        
        if epoch % 500 == 0:
            print("Epoch {}: loss = {}".format(epoch, loss))
    return params

In [20]:
t_un = 0.1 * t_u
params = training_loop(n_epochs=3000, learning_rate=1e-2,params=params, t_u=t_un, t_c=t_c)

Epoch 500: loss = 7.864030838012695
Epoch 1000: loss = 3.8292577266693115
Epoch 1500: loss = 3.0923218727111816
Epoch 2000: loss = 2.9577219486236572
Epoch 2500: loss = 2.9331398010253906
Epoch 3000: loss = 2.9286489486694336


In [28]:
def training_loop(n_epochs, optimizer, params, t_u, t_c):
    for epoch in range(1, n_epochs + 1 ):

        if params.grad is not None:
            params.grad.zero_()
        t_p = model (t_u, *params)
        loss = loss_fn(t_p, t_c)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print("Epoch {}: loss = {}".format(epoch, loss))
    return params

In [32]:
params = torch.tensor([1.0,0.0], requires_grad=True) 
learning_rate = 1e-2
optimizer = optim.SGD([params], lr=learning_rate)

t_p = model(t_u, *params)
loss = loss_fn(t_p, t_c)
loss.backward()
optimizer.step()
params

tensor([-44.1730,  -0.8260], requires_grad=True)

In [33]:
params = torch.tensor([1.0,0.0], requires_grad=True) 
learning_rate = 1e-2
optimizer = optim.SGD([params], lr=learning_rate)
training_loop(n_epochs=5000, optimizer=optimizer, params=params, t_u=t_un, t_c=t_c)

Epoch 500: loss = 7.860115051269531
Epoch 1000: loss = 3.828537940979004
Epoch 1500: loss = 3.092191219329834
Epoch 2000: loss = 2.957697868347168
Epoch 2500: loss = 2.933133840560913
Epoch 3000: loss = 2.9286484718322754
Epoch 3500: loss = 2.9278297424316406
Epoch 4000: loss = 2.9276793003082275
Epoch 4500: loss = 2.927651882171631
Epoch 5000: loss = 2.9276468753814697


tensor([  5.3671, -17.3012], requires_grad=True)